# 01 · Entrenamiento QLoRA del adaptador v1 (Colab T4)

**TraceGit · ISIA-119 · Grupo 3.** Especializa `google/gemma-4-E2B-it` para describir cambios de código (paso *map*)
con el dataset `brenpc/tracegit-commits`. Configuración de `PLAN.md` §4.3, validada en la prueba de humo
(`00_prueba_humo.ipynb`: 10,5 GB de VRAM pico y 1,2 s por ejemplo → unas 2 h para 2 épocas).

**El entrenamiento sobrevive a un corte de Colab:** cada 50 pasos el `Trainer` sube el último checkpoint a
`brenpc/tracegit-gemma4-e2b-lora-v1` (privado, subcarpeta `last-checkpoint`). Si la sesión se cae:
reconectar, **Run All**, y la celda 6 retoma desde ese checkpoint.

**Cómo ejecutarlo:** VS Code → *Select Kernel* → Colab → **GPU T4** → *Run All*. La celda 2 pide el token de
Hugging Face (con permiso de **escritura**) en un cuadro de texto: no pegarlo en el notebook ni en el chat.

La evaluación completa contra la línea base va en `02_evaluacion.ipynb`.

## 1. Configuración

In [1]:
MODELO = dict(repo="google/gemma-4-E2B-it", revision="4adf15748de9e1a0097b6274cee9c15836c41c37")
DATASET = dict(repo="brenpc/tracegit-commits", revision="8b5d79317fca720262bdb5334ed25e12442ef040")
REPO_ADAPTADOR = "brenpc/tracegit-gemma4-e2b-lora-v1"
SALIDA = "/content/adaptador_v1"

# PLAN.md §4.3
SEED = 42
MAX_LEN = 1536
EPOCAS = 2
BATCH, GRAD_ACCUM = 1, 16                 # lote efectivo 16
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
PASOS_GUARDADO = 50                       # guarda, evalúa y sube a Hub cada 50 pasos
N_VALIDACION = 120                        # subconjunto fijo de validación para la pérdida durante el entrenamiento

## 2. Dependencias y token de Hugging Face

Mismas versiones que la prueba de humo. Sin `datasets`: el JSONL se lee directo y se evita el conflicto de `pyarrow`.

In [2]:
%pip install -q "transformers==5.23.0" "peft==0.21.3" "bitsandbytes==0.50.2" "accelerate==1.16.0"

In [3]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Token de Hugging Face (permiso de escritura): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Sesión de Hugging Face:", whoami()["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Sesión de Hugging Face: brenpc


## 3. Entorno

In [4]:
import json, math, platform, random, time
import numpy as np
import torch, transformers, peft, bitsandbytes, accelerate

assert torch.cuda.is_available(), "No hay GPU: conectar el kernel a un servidor Colab con T4."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ENTORNO = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__,
}
print(json.dumps(ENTORNO, indent=2))

{
  "gpu": "Tesla T4",
  "vram_total_gb": 14.56,
  "python": "3.13.16",
  "torch": "2.14.1+cu130",
  "transformers": "5.23.0",
  "peft": "0.21.3",
  "bitsandbytes": "0.50.2",
  "accelerate": "1.16.0"
}


## 4. Dataset desde Hugging Face Hub (revisión fijada)

In [5]:
from huggingface_hub import hf_hub_download

def cargar(split):
    ruta = hf_hub_download(DATASET["repo"], f"{split}.jsonl", repo_type="dataset", revision=DATASET["revision"])
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

crudo = {s: cargar(s) for s in ("train", "validation", "test")}
print({s: len(v) for s, v in crudo.items()})
from collections import Counter
print("train por idioma:", Counter(x["meta"]["idioma"] for x in crudo["train"]))

train.jsonl:   0%|          | 0.00/8.74M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

{'train': 3080, 'validation': 459, 'test': 386}
train por idioma: Counter({'es': 1930, 'en': 1150})


## 5. Modelo en 4 bits, tokenización y enmascarado

Igual que en la prueba de humo: NF4 con *double quant* y cómputo en fp16 (la T4 no tiene bf16); las torres de visión y audio no se cuantizan; la pérdida se calcula solo sobre la respuesta, que termina en `<turn|>`.

In [6]:
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODELO["repo"], revision=MODELO["revision"])
if tok.chat_template is None:
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODELO["repo"], revision=MODELO["revision"]).chat_template
FIN_TURNO = "<turn|>"

import os, signal, subprocess

def vram_libre():
    return torch.cuda.mem_get_info()[0] / 2**30

def liberar_gpu():
    """Cierra los kernels viejos del mismo servidor (p. ej. la prueba de humo) que retienen la GPU. No toca este."""
    yo = os.getpid()
    sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
    print("Procesos en la GPU (pid, MiB):", sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader").strip() or "-")
    pids = {int(l.split()[0]) for l in sh("ps -eo pid,cmd | grep -E 'ipykernel|colab_kernel_launcher' | grep -v grep").splitlines()}
    pids |= {int(p) for p in sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader").split() if p.isdigit()}
    for pid in sorted(pids - {yo, os.getppid()}):
        try:
            print("  cerrando el proceso", pid, sh(f"ps -o cmd= -p {pid}").strip()[:80]); os.kill(pid, signal.SIGKILL)
        except (ProcessLookupError, PermissionError) as e:
            print("  no se pudo cerrar", pid, type(e).__name__)
    time.sleep(5)

if vram_libre() < 11:
    print(f"Solo {vram_libre():.1f} GB libres: hay otro kernel usando la GPU. Intentando liberarla...")
    liberar_gpu()
libre, total = (x / 2**30 for x in torch.cuda.mem_get_info())
print(f"VRAM libre: {libre:.1f} de {total:.1f} GB")
if libre < 11:
    raise RuntimeError("La GPU sigue ocupada por un proceso fuera de este servidor. Finalizar todas las sesiones de Colab "
                       "y conectar un servidor nuevo con T4.")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16,
                         llm_int8_skip_modules=["vision_tower", "audio_tower", "embed_vision", "embed_audio", "lm_head"])
t0 = time.time()
model = AutoModelForMultimodalLM.from_pretrained(MODELO["repo"], revision=MODELO["revision"],
                                                 quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
print(f"Modelo cargado en {time.time()-t0:.0f}s · VRAM {torch.cuda.memory_allocated()/2**30:.2f} GB")

def construir(ej):
    msgs = ej["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    completo = tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
    resp = completo[len(prompt):] if completo.startswith(prompt) else msgs[-1]["content"] + FIN_TURNO + "\n"
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(resp, add_special_tokens=False)["input_ids"]
    ids = (ids_p + ids_r)[:MAX_LEN]
    return {"input_ids": ids, "labels": ([-100] * len(ids_p) + ids_r)[:MAX_LEN], "truncado": len(ids_p) + len(ids_r) > MAX_LEN}

entreno = [construir(x) for x in crudo["train"]]
rng = random.Random(SEED)
validacion = [construir(x) for x in rng.sample(crudo["validation"], N_VALIDACION)]

fin = tok.decode([t for t in entreno[0]["labels"] if t != -100])
assert FIN_TURNO in fin, f"La respuesta no termina en {FIN_TURNO}"
lon = [len(x["input_ids"]) for x in entreno]
print(f"train {len(entreno)} · val {len(validacion)} · tokens media {np.mean(lon):.0f} p95 {np.percentile(lon,95):.0f} "
      f"máx {max(lon)} · truncados {sum(x['truncado'] for x in entreno)}")

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

VRAM libre: 14.5 de 14.6 GB


model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Modelo cargado en 126s · VRAM 6.28 GB
train 3080 · val 120 · tokens media 731 p95 1079 máx 1211 · truncados 0


## 6. LoRA y retoma desde Hugging Face Hub si hubo un corte

In [7]:
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi, snapshot_download

if hasattr(model, "peft_config"):
    raise RuntimeError("El modelo ya tiene LoRA (se re-ejecutó la celda). Reiniciar el kernel y ejecutar todo.")

model.activation_checkpointing_enable(activation_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
                  target_modules=r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)")
model = get_peft_model(model, lora)
for n, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
mods = {n.split(".lora_")[0] for n, _ in model.named_parameters() if "lora_" in n}
assert mods and not any(("vision" in m) or ("audio" in m) for m in mods)
print("módulos LoRA:", len(mods), "· entrenables:", sum(p.numel() for p in model.parameters() if p.requires_grad))

api = HfApi()
api.create_repo(REPO_ADAPTADOR, private=True, exist_ok=True)
RETOMAR = None
if any(f.startswith("last-checkpoint/") for f in api.list_repo_files(REPO_ADAPTADOR)):
    local = snapshot_download(REPO_ADAPTADOR, allow_patterns=["last-checkpoint/*"], local_dir=SALIDA)
    RETOMAR = os.path.join(local, "last-checkpoint")
    estado = json.load(open(os.path.join(RETOMAR, "trainer_state.json")))
    print(f"Se retoma desde el paso {estado['global_step']} (checkpoint en Hub).")
else:
    print("Entrenamiento nuevo (no hay checkpoint en Hub).")

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


módulos LoRA: 205 · entrenables: 24158208
Entrenamiento nuevo (no hay checkpoint en Hub).


## 7. Entrenamiento

Pérdida y gradiente por paso. En fp16 el `GradScaler` puede saltarse algún paso con gradiente `inf` al inicio (visto en la prueba de humo): es normal. Se detiene solo si la **pérdida** es NaN o si el gradiente es `inf` durante 5 pasos seguidos.

In [8]:
import inspect
from transformers import Trainer, TrainingArguments, TrainerCallback

acepta_tti = "token_type_ids" in inspect.signature(model.get_base_model().forward).parameters

def collate(lote):
    largo = max(len(x["input_ids"]) for x in lote)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.full((len(lote), largo), pad); lab = torch.full((len(lote), largo), -100)
    att = torch.zeros((len(lote), largo), dtype=torch.long)
    for i, x in enumerate(lote):
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"]); lab[i, :n] = torch.tensor(x["labels"]); att[i, :n] = 1
    out = {"input_ids": ids, "attention_mask": att, "labels": lab}
    if acepta_tti:
        out["token_type_ids"] = torch.zeros_like(ids)
    return out

class Vigia(TrainerCallback):
    def __init__(self): self.log, self.inf_seguidos = [], 0
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs:
            return
        if "eval_loss" in logs:
            self.log.append({"paso": state.global_step, "eval_loss": logs["eval_loss"]})
            print(f"  ── paso {state.global_step}: eval_loss {logs['eval_loss']:.4f}")
            return
        if "loss" not in logs:
            return
        loss, g = logs["loss"], logs.get("grad_norm")
        self.log.append({"paso": state.global_step, "loss": loss, "grad_norm": g, "lr": logs.get("learning_rate")})
        g_ok = g is None or math.isfinite(g)
        self.inf_seguidos = 0 if g_ok else self.inf_seguidos + 1
        if state.global_step % 10 == 0 or not g_ok:
            print(f"paso {state.global_step:>4}/{state.max_steps} | loss {loss:.4f} | grad_norm {g}")
        if not math.isfinite(loss) or self.inf_seguidos >= 5:
            print("FALLA: pérdida NaN o gradiente inf persistente. Se detiene."); control.should_training_stop = True

vigia = Vigia()
args = TrainingArguments(
    output_dir=SALIDA, num_train_epochs=EPOCAS, per_device_train_batch_size=BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=0.03,
    optim="paged_adamw_8bit", fp16=True, bf16=False, seed=SEED, logging_steps=1,
    eval_strategy="steps", eval_steps=PASOS_GUARDADO, save_strategy="steps", save_steps=PASOS_GUARDADO,
    save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    prediction_loss_only=True, push_to_hub=True, hub_model_id=REPO_ADAPTADOR, hub_private_repo=True,
    hub_strategy="checkpoint", report_to="none", remove_unused_columns=False, activation_checkpointing=False,
)
trainer = Trainer(model=model, args=args, train_dataset=entreno, eval_dataset=validacion,
                  data_collator=collate, callbacks=[vigia])

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
resultado = trainer.train(resume_from_checkpoint=RETOMAR)
HORAS = (time.time() - t0) / 3600
print(f"Fin: {HORAS:.2f} h en esta sesión · pico VRAM {torch.cuda.max_memory_allocated()/2**30:.2f} GB")
print(resultado)

Step,Training Loss,Validation Loss
50,3.041628,3.455837
100,2.053793,3.365865
150,2.459634,3.294068
200,2.353829,3.288748
250,2.245839,3.295308
300,2.278154,3.301620
350,1.814933,3.313618
386,2.338121,3.310646


paso    1/386 | loss 8.2111 | grad_norm nan
paso    2/386 | loss 7.7827 | grad_norm nan
paso    5/386 | loss 7.4039 | grad_norm nan
paso    9/386 | loss 5.7441 | grad_norm nan
paso   10/386 | loss 5.7316 | grad_norm 9.22214126586914
paso   12/386 | loss 5.5337 | grad_norm nan
paso   20/386 | loss 3.2494 | grad_norm 5.73457145690918
paso   30/386 | loss 3.2367 | grad_norm 5.06040096282959
paso   40/386 | loss 3.1901 | grad_norm 2.6064229011535645
paso   50/386 | loss 3.0416 | grad_norm 3.1746022701263428
  ── paso 50: eval_loss 3.4558
paso   60/386 | loss 2.6200 | grad_norm 2.253916025161743
paso   70/386 | loss 3.6814 | grad_norm 3.27398419380188
paso   80/386 | loss 1.8008 | grad_norm 2.2917494773864746
paso   90/386 | loss 2.7554 | grad_norm 2.1766357421875
paso  100/386 | loss 2.0538 | grad_norm 1.978883981704712
  ── paso 100: eval_loss 3.3659
paso  110/386 | loss 3.1130 | grad_norm 2.3656208515167236
paso  120/386 | loss 2.6042 | grad_norm 3.2982847690582275
paso  130/386 | loss 2

## 8. Guardar y publicar el adaptador final con su *model card*

In [9]:
trainer.save_model(SALIDA)            # con load_best_model_at_end es el de menor eval_loss
historial = trainer.state.log_history
evals = [(h["step"], round(h["eval_loss"], 4)) for h in historial if "eval_loss" in h]
mejor = trainer.state.best_metric

REGISTRO = {
    "modelo_base": f"{MODELO['repo']}@{MODELO['revision']}", "dataset": f"{DATASET['repo']}@{DATASET['revision']}",
    "ejemplos_train": len(entreno), "ejemplos_val_usados": len(validacion), "epocas": EPOCAS,
    "pasos": trainer.state.global_step, "lote_efectivo": BATCH * GRAD_ACCUM, "lr": LR,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "modulos": len(mods)},
    "eval_loss_por_paso": evals, "mejor_eval_loss": mejor, "mejor_checkpoint": trainer.state.best_model_checkpoint,
    "horas_ultima_sesion": round(HORAS, 2), "pico_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "entorno": ENTORNO, "fecha": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
}
with open(os.path.join(SALIDA, "registro_entrenamiento.json"), "w", encoding="utf-8") as f:
    json.dump({**REGISTRO, "log": historial}, f, ensure_ascii=False, indent=2)

filas = "\n".join(f"| {p} | {l} |" for p, l in evals)
CARD = f"""---
base_model: {MODELO['repo']}
library_name: peft
license: apache-2.0
language: [es, en]
datasets: [{DATASET['repo']}]
tags: [lora, qlora, git, commits, code]
---

# TraceGit · adaptador QLoRA v1 para Gemma 4 E2B-it

Adaptador LoRA que especializa `{MODELO['repo']}` (revisión `{MODELO['revision'][:7]}`) en **describir cambios de
código**: dado un diff de git, escribe un mensaje de commit. Es el paso *map* de TraceGit (proyecto integrador
ISIA-119, Grupo 3). Código: https://github.com/brenpc1443/tracegit

**Uso previsto:** paso interno de un sistema que genera informes de proceso con evidencia citada.
**No** sirve para decidir si un código lo escribió una persona o una IA.

## Entrenamiento

| | |
|---|---|
| Dataset | `{DATASET['repo']}` @ `{DATASET['revision'][:7]}`: {len(entreno)} ejemplos de entrenamiento (commits humanos anteriores al 30-11-2022) |
| Método | QLoRA: base en NF4 con *double quant*, cómputo fp16; pérdida solo sobre la respuesta |
| LoRA | r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}; q,k,v,o,gate,up,down del modelo de lenguaje ({len(mods)} módulos) |
| Optimización | paged_adamw_8bit, lr {LR}, coseno, warmup 3 %, lote efectivo {BATCH*GRAD_ACCUM}, {EPOCAS} épocas, semilla {SEED} |
| Hardware | {ENTORNO['gpu']} (Colab gratuito) |
| Selección | checkpoint de menor pérdida en validación ({N_VALIDACION} ejemplos fijos): **{mejor}** |

| Paso | eval_loss |
|---|---|
{filas}

## Limitaciones

- La métrica de pérdida no mide la fidelidad al diff: la evaluación con ROUGE-L, exactitud de identificadores y
  brecha por idioma está en `notebooks/02_evaluacion.ipynb` del repositorio.
- Hereda los sesgos del modelo base y del dataset (62 % de mensajes en español, proyectos tipo curso).
"""
with open(os.path.join(SALIDA, "README.md"), "w", encoding="utf-8") as f:
    f.write(CARD)

from huggingface_hub import upload_folder
info = upload_folder(repo_id=REPO_ADAPTADOR, folder_path=SALIDA,
                     allow_patterns=["adapter_model.safetensors", "adapter_config.json", "README.md",
                                     "registro_entrenamiento.json", "tokenizer*", "chat_template*"],
                     commit_message=f"Adaptador v1 final (mejor eval_loss {mejor})")
print("Publicado:", f"https://huggingface.co/{REPO_ADAPTADOR}", "@", info.oid)
print(json.dumps(REGISTRO, indent=2, ensure_ascii=False))

Publicado: https://huggingface.co/brenpc/tracegit-gemma4-e2b-lora-v1 @ 9157fe0c5e22b8ce6a02723d2f6e1c9c23694636
{
  "modelo_base": "google/gemma-4-E2B-it@4adf15748de9e1a0097b6274cee9c15836c41c37",
  "dataset": "brenpc/tracegit-commits@8b5d79317fca720262bdb5334ed25e12442ef040",
  "ejemplos_train": 3080,
  "ejemplos_val_usados": 120,
  "epocas": 2,
  "pasos": 386,
  "lote_efectivo": 16,
  "lr": 0.0002,
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "modulos": 205
  },
  "eval_loss_por_paso": [
    [
      50,
      3.4558
    ],
    [
      100,
      3.3659
    ],
    [
      150,
      3.2941
    ],
    [
      200,
      3.2887
    ],
    [
      250,
      3.2953
    ],
    [
      300,
      3.3016
    ],
    [
      350,
      3.3136
    ],
    [
      386,
      3.3106
    ]
  ],
  "mejor_eval_loss": 3.288748264312744,
  "mejor_checkpoint": "/content/adaptador_v1/checkpoint-200",
  "horas_ultima_sesion": 2.62,
  "pico_vram_gb": 10.84,
  "entorno": {
    "gpu": 

## 9. Comprobación rápida: base frente a adaptador (5 ejemplos de prueba)

Solo para ver que el adaptador cambia la salida en la dirección esperada. Las métricas van en el notebook 02.

In [ ]:
model.eval(); model.config.use_cache = True

def generar(msgs, adaptador=True, max_new=64):
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        if adaptador:
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        else:
            with model.disable_adapter():
                out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

for x in random.Random(1).sample(crudo["test"], 5):
    print(f"[{x['meta']['idioma']} · {x['meta']['lenguaje']}]")
    print("  HUMANO   :", x["messages"][-1]["content"][:150])
    print("  BASE     :", generar(x["messages"], adaptador=False)[:150].replace("\n", " "))
    print("  ADAPTADOR:", generar(x["messages"], adaptador=True)[:150].replace("\n", " "))

[en · Python]
  HUMANO   : Eliminar warnings en la api
  BASE     : **Commit Message:**  Fix: Correct return status code for successful message creation in `incoming_message` view.
  ADAPTADOR: Add files via upload
[es · TypeScript]
  HUMANO   : Mejore un poco la home.

+ Nuevos botones.
+ Aumente el espacio a los costados en todas las pantallas y en la
navbar.
+ Agregado boton seguir usuario.
  BASE     : **feat: Refactorización de estilos y estructura en componentes de creación y home**  Se realizaron los siguientes cambios:  *   **`client/src/app/crea
  ADAPTADOR: fix: create-rv
[en · Python]
  HUMANO   : added models -- could change in future
  BASE     : feat: Implement basic models for Usuario and Numero in CLI/Bingo/models.py  This commit introduces the `Usuario` and `Numero` dataclasses in `CLI/Bing
  ADAPTADOR: Add files via upload
[en · TypeScript]
  HUMANO   : less redundant bugfix
  BASE     : **feat: Renamed `setColor` to `changeColor` in Receptor classes**  Refactors the 

: 